# Bước 5: Model building – xây dựng và huấn luyện mô hình

Nguyên tắc: **không chọn mô hình theo cảm tính**. Ta thử nhiều ứng viên, đo bằng *kiểm tra chéo* (cross-validation) trên tập train, rồi chọn cái tốt nhất.
Tập test chưa được đụng tới, để dành cho bước 6.

Mã thật nằm ở `pipelines/training/nodes.py`; chạy và lưu bằng `kedro run`.

In [ ]:
import importlib.util
import logging
import sys
import warnings
from pathlib import Path

# Kiểm tra kernel đã có đủ thư viện chưa, để báo lỗi dễ hiểu nếu thiếu
required = ["kedro", "kedro_datasets", "pandas", "pyarrow", "scipy", "sklearn", "matplotlib"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ModuleNotFoundError(
        f"Kernel này ({sys.executable}) thiếu: {missing}.\n"
        "Chạy trong terminal rồi khởi động lại kernel:\n"
        f'  "{sys.executable}" -m pip install -r requirements.txt'
    )

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project


def find_project_root() -> Path:
    # Tìm thư mục gốc của dự án Kedro, dù notebook được mở từ đâu
    here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "x")).resolve().parent
    for folder in (here, *here.parents, Path.cwd(), *Path.cwd().parents):
        if (folder / "conf").is_dir() and (folder / "pyproject.toml").exists():
            return folder
    for pyproject in Path.cwd().glob("*/pyproject.toml"):  # mở từ thư mục cha
        if "[tool.kedro]" in pyproject.read_text(encoding="utf-8"):
            return pyproject.parent
    raise FileNotFoundError("Không tìm thấy dự án Kedro (thư mục có conf/ và pyproject.toml)")


project_path = find_project_root()
bootstrap_project(project_path)  # thêm thư mục src vào đường dẫn import
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    catalog = context.catalog  # dữ liệu khai báo trong conf/base/catalog.yml
    params = context.params    # tham số trong conf/base/parameters*.yml

for name in ("kedro", "netflix"):  # bớt log của Kedro trong notebook
    logging.getLogger(name).setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Python đang dùng:", sys.executable)
print("Thư mục dự án:", project_path)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

from netflix.pipelines.data_processing.nodes import clean_data
from netflix.pipelines.feature_engineering.nodes import create_features, split_data
from netflix.pipelines.training.nodes import _make_pipeline, compare_models, train_best_model

fe = params["feature_engineering"]
model_options = params["model_options"]
training = params["training"]

clean = clean_data(catalog.load("netflix_raw"), params["data_processing"])
X_train, X_test, y_train, y_test = split_data(create_features(clean, fe), fe, model_options)
print("Dùng để huấn luyện:", X_train.shape, "| cột:", list(X_train.columns))

## 5.1 Mốc so sánh (baseline)
Trước khi khen một mô hình, hỏi: *nó hơn cách đoán ngớ ngẩn bao nhiêu?*
`DummyClassifier` luôn đoán lớp phổ biến nhất. ROC-AUC = 0,5 nghĩa là không hơn tung đồng xu.

In [ ]:
folds = StratifiedKFold(n_splits=training["cv_folds"], shuffle=True, random_state=model_options["random_state"])
baseline = cross_val_score(DummyClassifier(strategy="prior"), X_train, y_train.iloc[:, 0], cv=folds, scoring="roc_auc")
print(f"Baseline ROC-AUC: {baseline.mean():.3f}")

## 5.2 So sánh các ứng viên
Danh sách ứng viên và siêu tham số nằm trong `conf/base/parameters_training.yml`.
Cách đọc bảng:
- `cv_score_mean`: điểm trung bình trên các phần kiểm tra. Càng cao càng tốt.
- `cv_score_std`: độ dao động giữa các lần. Nhỏ nghĩa là ổn định.
- `overfit_gap`: điểm trên dữ liệu đã học trừ điểm trên dữ liệu chưa thấy. Lớn nghĩa là mô hình **học thuộc lòng**.

In [ ]:
comparison = compare_models(X_train, y_train, model_options, training)
comparison

Các mô hình có điểm gần nhau (chênh dưới độ dao động `cv_score_std`) thì **chưa thể nói cái nào thật sự hơn**; khi đó chọn mô hình đơn giản hơn, dễ giải thích hơn.

## 5.3 Thử chỉnh một siêu tham số
Siêu tham số là các núm vặn do ta đặt, không do mô hình tự học. Ví dụ rừng ngẫu nhiên có `max_depth` (độ sâu cây): quá nông thì không học đủ, quá sâu thì học thuộc lòng.

In [ ]:
base = {**training["candidates"]["random_forest"], "n_estimators": 100}
rows = []
for depth in [2, 4, 6, 8, 12, 20]:
    model = _make_pipeline(X_train, "random_forest", {**base, "max_depth": depth}, model_options["random_state"])
    scores = cross_val_score(model, X_train, y_train.iloc[:, 0], cv=folds, scoring="roc_auc")
    rows.append({"max_depth": depth, "cv_roc_auc": scores.mean().round(4), "std": scores.std().round(4)})
pd.DataFrame(rows)

Tìm điểm mà tăng `max_depth` không còn cải thiện nữa. Nếu bạn muốn dùng giá trị khác, sửa trong `parameters_training.yml`.

## 5.4 Huấn luyện mô hình tốt nhất
`train_best_model` lấy ứng viên điểm cao nhất rồi huấn luyện lại trên **toàn bộ** tập train.

In [ ]:
classifier = train_best_model(X_train, y_train, comparison, model_options, training)
print("Mô hình được chọn:", type(classifier.named_steps["model"]).__name__)
print("Số cột đầu vào:", len(classifier.feature_names_in_), "->", list(classifier.feature_names_in_))

> Mô hình này **chưa** được chấm trên tập test. Việc đó là của bước 6, và chỉ làm **một lần**, để điểm số không bị tối ưu theo tập test.

## Ghi chú của bạn
Viết vài dòng bằng lời của bạn: bạn học được gì, điều gì bất ngờ, bạn sẽ thử gì tiếp?

-
-
-